# 12 — Tool Calling
### LangChain Agents & Tools · Part 2 of 3 (`11_Tools` → `12_Tool_Calling` → `13_Agents`)

In `11_Tools` you built the menu and the kitchen, then *played the model by hand*, writing the tool
calls yourself. Now the real model arrives. This notebook is the loop that connects the three:

```
 You: "Can I get a refund for order A-1001?"
        │
        ▼
  MODEL (with tools bound) ── reads the menu, writes a tool call. It does NOT run anything.
        │
        ▼
  AI message:  tool_calls = [{"name": "check_refund_eligibility", "args": {"order_id": "A-1001"}, "id": "call_1"}]
        │
        ▼
  YOUR CODE runs the tool
        │
        ▼
  Tool message: "Eligible: delivered 10 days ago..."   (tagged with the call's id)
        │
        ▼
  MODEL again ── reads the result ── another tool call, or a final answer
```

That loop, repeated until the model stops asking for tools, is the whole idea. An **agent** (next
notebook) is simply this loop, packaged. Writing it yourself first means the agent won't be a black box.

> 📌 **What was checked while building this notebook**
> - **A malformed tool call can look like "the model is done."** With `.invoke`, if the model writes
>   arguments that aren't valid JSON, LangChain puts the call in `invalid_tool_calls` and leaves
>   `tool_calls` **empty**. A naive loop (`if not ai.tool_calls: return ai.content`) then returns a
>   blank answer with no error. Section 4's loop handles this.
> - **A *streamed* malformed call can look valid.** While streaming, a truncated
>   `{"a": 20, "b": ` is quietly repaired into `{"a": 20}` — a plausible-looking call that is missing an
>   argument. Section 5 shows it, and why your tools' input validation is the second safety net.
> - **Groq can reject a bad tool call outright** with `400 tool_use_failed`. Forum and issue-tracker
>   reports (not an official Groq statement) say this happens most on the small `llama-3.1-8b-instant`
>   and on Llama 4 models, and least on `llama-3.3-70b-versatile` — so that is the model used here.
>   Groq's API has also been reported to reject calls whose arguments don't match your schema
>   (`tool call validation failed`), so expect that error shape too.
> - `bind_tools` options differ by provider (read from the installed packages): `ChatGroq` accepts a
>   tool name, `"auto"`, `"any"`, or a dict as `tool_choice`; `ChatOpenAI` additionally accepts
>   `"none"` and `"required"`, plus `strict` and `parallel_tool_calls`.
> - **Not run live:** the sandbox this notebook was built in couldn't reach Groq. The loop, the failure
>   handling and the guardrails were executed against stand-in models (Section 4 builds one
>   *inside* the notebook, so you can reproduce the failures on demand). What a real model chooses to
>   do on your questions is for you to observe.

## Setup

In [ ]:
# %pip install -q langchain-core langchain-groq python-dotenv pydantic
# %pip install -q langchain-openai        # optional: only if you swap in OpenAI (end of Section 6)


In [1]:
import os, json
from datetime import date, timedelta
from typing import Callable

from dotenv import load_dotenv
load_dotenv()

from langchain_core.tools import tool, ToolException
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, AIMessageChunk, ToolMessage
from langchain_groq import ChatGroq

print("Ready. Needs GROQ_API_KEY in your .env for the live cells.")


Ready. Needs GROQ_API_KEY in your .env for the live cells.


---
## 1. Bind tools to a model, and read what comes back

Start with the two tools from your first script: `add` and `multiply`. `bind_tools` gives the model the
menu — and returns a *new* model object that includes it. The original `llm` is unchanged.

In [6]:
@tool(parse_docstring=True)
def add(a: int, b: int) -> int:
    """Add two integers and return the sum.

    Args:
        a: The first number.
        b: The second number.
    """
    return a + b

@tool(parse_docstring=True)
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the product.

    Args:
        a: The first number.
        b: The second number.
    """
    return a * b

llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)
llm_with_tools = llm.bind_tools([add, multiply])
print("Model ready, with 2 tools bound.")


Model ready, with 2 tools bound.


In [7]:
ai = llm_with_tools.invoke("What is 20 plus 20?")

print("type      :", type(ai).__name__)
print("content   :", repr(ai.content))          # usually empty: the model is asking, not answering
print("tool_calls:", ai.tool_calls)


type      : AIMessage
content   : ''
tool_calls: [{'name': 'add', 'args': {'a': 20, 'b': 20}, 'id': '4gh8brnz6', 'type': 'tool_call'}]


You should see an `AIMessage` whose `content` is empty and whose `tool_calls` holds something shaped
like `[{'name': 'add', 'args': {'a': 20, 'b': 20}, 'id': '...', 'type': 'tool_call'}]`. The `id` matters:
the result you send back must carry the same id, so the model can match answers to requests.

**The model decides whether to use a tool at all.** Ask something that needs none:

In [8]:
ai = llm_with_tools.invoke("Say hello in one short sentence.")
print("content   :", ai.content)
print("tool_calls:", ai.tool_calls)             # [] — no tool needed, so it just answered


content   : Hello!
tool_calls: []


---
## 2. One full round, by hand

This is your original script's flow, with two small changes: the model is named explicitly (your script
used `ChatOpenAI()` with no model, so the model depended on whatever default your installed version
picks), and `tool.invoke(tool_call)` is given the whole call so it hands back a ready-made `ToolMessage`.

The order of messages matters and providers enforce it: **the AI message containing the tool calls must
come before the tool messages that answer it**, and each tool message needs the matching `tool_call_id`.

In [9]:
tools = [add, multiply]
tool_map = {t.name: t for t in tools}

messages = [HumanMessage("What is 20 plus 20?")]

ai = llm_with_tools.invoke(messages)
messages.append(ai)                                   # 1. keep the AI message that asked for the tool

for call in ai.tool_calls:
    result = tool_map[call["name"]].invoke(call)      # 2. YOUR code runs the tool -> a ToolMessage
    messages.append(result)

final = llm_with_tools.invoke(messages)               # 3. the model reads the result and answers
print(final.content)
print("\nThe conversation was:", [type(m).__name__ for m in messages])


20 plus 20 equals **40**.

The conversation was: ['HumanMessage', 'AIMessage', 'ToolMessage']


---
## 3. Parallel tool calls — several requests in one message

When two requests don't depend on each other, a model can ask for both at once. Each call arrives in the
same `AIMessage` with its own id, and **every one needs its own `ToolMessage`** before you call the model
again.

In [10]:
messages = [HumanMessage("What is 3 times 4, and what is 10 plus 5?")]

ai = llm_with_tools.invoke(messages)
messages.append(ai)
print(f"The model asked for {len(ai.tool_calls)} tool call(s) in one message:")
for call in ai.tool_calls:
    print("  ", call["name"], call["args"], "| id:", call["id"])
    messages.append(tool_map[call["name"]].invoke(call))

print("\nFinal:", llm_with_tools.invoke(messages).content)


The model asked for 2 tool call(s) in one message:
   multiply {'a': 3, 'b': 4} | id: b5q1esxsx
   add {'a': 10, 'b': 5} | id: z65b3qry4

Final: 3 times 4 is **12**, and 10 plus 5 is **15**.


(With `ChatOpenAI` you can turn this off with `bind_tools(..., parallel_tool_calls=False)` — a real
option in its signature. `ChatGroq`'s `bind_tools` has no such argument.)

Calls that **depend** on each other can't be parallel: "add 20 and 20, *then* multiply the result by 3"
needs the first answer before the second call can even be written. That takes two rounds — which is why
you need a loop.

---
## 4. The loop — and making it safe

Everything above handled exactly one round. Real questions need an unknown number of rounds, and real
systems fail. So the loop below does five things on purpose:

| Protection | Why |
|---|---|
| **`max_steps`** | A confused model can keep asking for tools forever. This is the circuit breaker |
| **Errors become messages** | An unknown tool name or bad arguments go back to the model as text, so it can correct itself, instead of crashing the run |
| **Retry malformed calls** | Both failure shapes from the callout above: a provider `tool_use_failed` error, and `invalid_tool_calls` with an empty `tool_calls` |
| **Approval gate** | Tools listed in `needs_approval` run only if a human says yes. **Default is deny** |
| **Answer every call** | Every tool call gets a `ToolMessage`, or the next model call is rejected |

In [11]:
def execute_tool_call(call: dict, tool_map: dict) -> ToolMessage:
    """Run one tool call. Whatever goes wrong comes back as a ToolMessage the model can read."""
    name = call["name"]
    if name not in tool_map:
        return ToolMessage(
            content=f"Error: there is no tool named '{name}'. Available tools: {', '.join(tool_map)}.",
            tool_call_id=call["id"], name=name, status="error",
        )
    try:
        return tool_map[name].invoke(call)
    except Exception as e:                      # bad arguments, ToolException, anything else
        detail = " | ".join(line.strip() for line in str(e).splitlines()[:3]) or type(e).__name__
        return ToolMessage(content=f"Error ({type(e).__name__}): {detail}",
                           tool_call_id=call["id"], name=name, status="error")


def invoke_with_retry(llm_with_tools, messages: list, attempts: int = 3) -> AIMessage:
    """Call the model, retrying the two ways a tool call can come back broken."""
    for attempt in range(1, attempts + 1):
        last_try = attempt == attempts
        try:
            ai = llm_with_tools.invoke(messages)
        except Exception as e:
            if "tool_use_failed" in str(e) and not last_try:      # e.g. Groq's 400 on a malformed call
                print(f"  (provider rejected a malformed tool call; retry {attempt}/{attempts - 1})")
                continue
            raise
        if ai.invalid_tool_calls and not ai.tool_calls:           # unparseable arguments, NOT "finished"
            if last_try:
                raise RuntimeError(f"Model kept writing unparseable tool arguments: {ai.invalid_tool_calls}")
            print(f"  (model wrote unparseable tool arguments; retry {attempt}/{attempts - 1})")
            continue
        return ai


def run_tool_loop(
    llm_with_tools, messages: list, tools: list, *,
    max_steps: int = 6,
    needs_approval: frozenset = frozenset(),
    approve: Callable[[dict], bool] | None = None,
    verbose: bool = True,
) -> str:
    """Run the call -> execute -> return loop until the model answers. Mutates `messages` in place,
    so the same list can carry on into the next user turn."""
    tool_map = {t.name: t for t in tools}
    for step in range(1, max_steps + 1):
        ai = invoke_with_retry(llm_with_tools, messages)
        messages.append(ai)
        if not ai.tool_calls:
            return ai.content                                  # the model is done

        for call in ai.tool_calls:
            if verbose:
                print(f"  step {step}: {call['name']}({json.dumps(call['args'])})")
            if call["name"] in needs_approval and not (approve and approve(call)):
                result = ToolMessage(
                    content=("This action was declined by a human reviewer and was NOT performed. "
                             "Do not retry it. Explain this to the user and offer to escalate to a human."),
                    tool_call_id=call["id"], name=call["name"], status="error",
                )
            else:
                result = execute_tool_call(call, tool_map)
            if verbose:
                print(f"           -> {result.content}")
            messages.append(result)

    return f"[stopped after {max_steps} steps without a final answer]"


### The loop on a question that needs two rounds

"Add 20 and 20, then multiply the result by 3" — the second call depends on the first answer. The
`step` numbers in the trace show the rounds.

In [12]:
messages = [HumanMessage("Add 20 and 20, then multiply the result by 3.")]
answer = run_tool_loop(llm_with_tools, messages, tools)
print("\nFinal answer:", answer)


  step 1: add({"a": 20, "b": 20})
           -> 40
  step 2: multiply({"a": 40, "b": 3})
           -> 120

Final answer: The result is **120**.

(20 + 20 = 40, then 40 × 3 = 120)


### Making it fail on purpose

Real models won't misbehave on cue, so you can't easily *see* these protections work. The cell below
builds a tiny **scripted stand-in model** that replays a fixed list of replies — including broken ones.
It has the same interface as a real model (`bind_tools`, `invoke`), so the loop can't tell the
difference. This is also how you'd unit-test an agent without paying for API calls.

In [13]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatGeneration, ChatResult


class ScriptedModel(BaseChatModel):
    """A fake chat model that replays a fixed list of AIMessages (the last one repeats forever)."""
    script: list
    position: int = 0
    raise_first: int = 0            # raise a provider-style error this many times before answering

    @property
    def _llm_type(self) -> str:
        return "scripted"

    def bind_tools(self, tools, **kwargs):
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        if self.raise_first > 0:
            self.raise_first -= 1
            raise RuntimeError("Error code: 400 - tool_use_failed: Failed to call a function.")
        reply = self.script[min(self.position, len(self.script) - 1)]
        self.position += 1
        return ChatResult(generations=[ChatGeneration(message=reply)])


def make_call(name, args, cid):
    return {"name": name, "args": args, "id": cid, "type": "tool_call"}

def demo(title, model, question="What is 20 plus 20?", **loop_kwargs):
    print(f"\n=== {title} ===")
    msgs = [HumanMessage(question)]
    print("  result:", run_tool_loop(model.bind_tools(tools), msgs, tools, **loop_kwargs))


In [14]:
demo("A. The provider rejects a malformed call once, then it works",
     ScriptedModel(raise_first=1, script=[
         AIMessage(content="", tool_calls=[make_call("add", {"a": 20, "b": 20}, "c1")]),
         AIMessage(content="20 plus 20 is 40."),
     ]))

demo("B. The model writes unparseable arguments, then fixes it",
     ScriptedModel(script=[
         AIMessage(content="", invalid_tool_calls=[{"type": "invalid_tool_call", "name": "add", "id": "c1",
                                                    "args": '{"a": 20, "b": ', "error": "not valid JSON"}]),
         AIMessage(content="", tool_calls=[make_call("add", {"a": 20, "b": 20}, "c2")]),
         AIMessage(content="20 plus 20 is 40."),
     ]))

demo("C. The model asks for a tool that does not exist",
     ScriptedModel(script=[
         AIMessage(content="", tool_calls=[make_call("subtract", {"a": 20, "b": 20}, "c1")]),
         AIMessage(content="Sorry, I can only add and multiply."),
     ]))

demo("D. The model forgets a required argument, then corrects itself",
     ScriptedModel(script=[
         AIMessage(content="", tool_calls=[make_call("add", {"a": 20}, "c1")]),
         AIMessage(content="", tool_calls=[make_call("add", {"a": 20, "b": 20}, "c2")]),
         AIMessage(content="20 plus 20 is 40."),
     ]))

demo("E. A runaway model that never stops asking (the circuit breaker)",
     ScriptedModel(script=[AIMessage(content="", tool_calls=[make_call("add", {"a": 1, "b": 1}, "c1")])]),
     max_steps=3)



=== A. The provider rejects a malformed call once, then it works ===
  (provider rejected a malformed tool call; retry 1/2)
  step 1: add({"a": 20, "b": 20})
           -> 40
  result: 20 plus 20 is 40.

=== B. The model writes unparseable arguments, then fixes it ===
  (model wrote unparseable tool arguments; retry 1/2)
  step 1: add({"a": 20, "b": 20})
           -> 40
  result: 20 plus 20 is 40.

=== C. The model asks for a tool that does not exist ===
  step 1: subtract({"a": 20, "b": 20})
           -> Error: there is no tool named 'subtract'. Available tools: add, multiply.
  result: Sorry, I can only add and multiply.

=== D. The model forgets a required argument, then corrects itself ===
  step 1: add({"a": 20})
           -> Error (ValidationError): 1 validation error for add | b | Field required [type=missing, input_value={'a': 20}, input_type=dict]
  step 2: add({"a": 20, "b": 20})
           -> 40
  result: 20 plus 20 is 40.

=== E. A runaway model that never stops asking 

Read the traces. In **C** and **D** the error text went back to the model, which then recovered — no
crash. In **B** the empty `tool_calls` did *not* fool the loop. In **E**, without `max_steps` this would
never return. (A real model with a real bug can behave exactly like E, and you'd be paying per call.)

---
## 5. Streaming tool calls — with one trap

When you stream, a tool call arrives in **fragments**. Chunks add together (`+`), and the argument text
builds up piece by piece. Everything below runs offline with hand-made chunks.

In [15]:
chunks = [
    AIMessageChunk(content="", tool_call_chunks=[{"name": "add", "args": "",           "id": "call_1", "index": 0}]),
    AIMessageChunk(content="", tool_call_chunks=[{"name": None,  "args": '{"a": 20,',  "id": None,     "index": 0}]),
    AIMessageChunk(content="", tool_call_chunks=[{"name": None,  "args": ' "b": 20}',  "id": None,     "index": 0}]),
]

merged = chunks[0]
for c in chunks[1:]:
    merged = merged + c
    print("args text so far:", merged.tool_call_chunks[0]["args"])

print("\nfinished tool_calls:", merged.tool_calls)


args text so far: {"a": 20,
args text so far: {"a": 20, "b": 20}

finished tool_calls: [{'name': 'add', 'args': {'a': 20, 'b': 20}, 'id': 'call_1', 'type': 'tool_call'}]


**The trap:** while a call is still arriving — or if the stream is cut off — LangChain repairs the
half-written JSON so partial results display nicely. Watch what happens to a call that was cut off
mid-argument:

In [16]:
truncated = AIMessageChunk(content="", tool_call_chunks=[
    {"name": "add", "args": '{"a": 20, "b": ', "id": "call_9", "index": 0}   # the stream died here
])

print("tool_calls:", truncated.tool_calls)          # looks like a valid call... but `b` has vanished
print("invalid   :", truncated.invalid_tool_calls)  # empty: nothing flagged it

try:
    add.invoke(truncated.tool_calls[0])
except Exception as e:
    print("\nThe tool's own validation catches it ->", " | ".join(l.strip() for l in str(e).splitlines()[:3]))


tool_calls: [{'name': 'add', 'args': {'a': 20}, 'id': 'call_9', 'type': 'tool_call'}]
invalid   : []

The tool's own validation catches it -> 1 validation error for add | b | Field required [type=missing, input_value={'a': 20}, input_type=dict]


Two rules follow. **Only act on a tool call after the stream has finished** (partial arguments are
provisional). And **let your tools validate their inputs** — type hints and Pydantic schemas (from
`11_Tools`) are your second safety net, and here one just caught a call the parser had silently mangled.

The live pattern for streaming from a real model is the same merge-as-you-go:

In [17]:
gathered = None
for chunk in llm_with_tools.stream("What is 7 times 8?"):
    gathered = chunk if gathered is None else gathered + chunk

print(gathered.tool_calls)        # trust this only now that the stream has ended


[{'name': 'multiply', 'args': {'a': 7, 'b': 8}, 'id': '708xatek9', 'type': 'tool_call'}]


---
## 6. `tool_choice` — telling the model whether it must use a tool

By default the model decides (`"auto"`). Sometimes you want to take that decision away:

| Setting | Meaning | When |
|---|---|---|
| `"auto"` (default) | Model chooses: a tool, several, or none | Normal assistants |
| `"any"` | **Must** call some tool | Routing: "pick the right action for this message" |
| `"<tool name>"` | **Must** call that exact tool | Extraction: force structured output through one tool |
| `"none"` (OpenAI) | Must **not** call tools | A final "summarise" step |

(Checked against the installed `ChatGroq`: binding two tools with a tool name works, and `"any"` is
sent to Groq as `"required"`. Whether a provider honors `"none"` is the provider's side, so test it.)

In [18]:
must_use_a_tool  = llm.bind_tools(tools, tool_choice="any")
must_use_multiply = llm.bind_tools(tools, tool_choice="multiply")

print("any      :", must_use_a_tool.invoke("What is 6 times 7?").tool_calls)
print("multiply :", must_use_multiply.invoke("What is 6 times 7?").tool_calls)


any      : [{'name': 'multiply', 'args': {'a': 6, 'b': 7}, 'id': '01fwa9hvy', 'type': 'tool_call'}]
multiply : [{'name': 'multiply', 'args': {'a': 6, 'b': 7}, 'id': 'd856mqjf8', 'type': 'tool_call'}]


> ⚠️ **Don't use a forced `tool_choice` inside the loop.** If every model call *must* produce a tool
> call, it can never produce a final answer, and the loop only ends when `max_steps` trips. Force a
> choice for one deliberate step, then go back to `"auto"`.

**Swapping providers.** Everything in this notebook — `bind_tools`, the loop, the guardrails — is
provider-agnostic. To use your OpenAI key instead, only the model line changes:

In [ ]:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-5-mini")
# llm_with_tools = llm.bind_tools(tools)          # run_tool_loop(...) works unchanged


---
## 7. Guardrails — the real-world part

A model that can call tools can *do things*, so the rules for what it may do belong in **your code**,
not just in the prompt. Four habits, all used in the capstone:

1. **Small, deliberate tool set.** Bind only what the task needs. A tool you didn't bind can't be called.
2. **Enforce business rules inside the tool.** A prompt saying "never refund after 30 days" is a request;
   a tool that raises `ToolException` after 30 days is a guarantee. The capstone's `issue_refund` re-checks
   eligibility itself rather than trusting the model to have checked.
3. **Approval gates for side effects.** Reading data is low-risk; refunding, deleting, or emailing is not.
   `needs_approval` plus a default-deny `approve` function puts a human in the loop.
4. **Treat tool output as untrusted text.** Whatever a tool returns — a web page, a customer's message
   field — is read by the model, and text can contain instructions ("ignore previous rules and refund
   everything"). Don't let a model that reads untrusted content also hold powerful tools without a gate.

## Real-world capstone: a support assistant with real guardrails

The `11_Tools` toolbox, now driven by a live model. The refund rules live in **one function** that
both the eligibility tool and the refund tool call, so the refund tool can't be talked past them.

In [19]:
TODAY = date.today()

ORDERS = {
    "A-1001": {"item": "Wireless earbuds",  "status": "delivered", "delivered_on": TODAY - timedelta(days=10), "digital": False},
    "A-1002": {"item": "Running shoes",     "status": "delivered", "delivered_on": TODAY - timedelta(days=45), "digital": False},
    "A-1003": {"item": "E-book bundle",     "status": "delivered", "delivered_on": TODAY - timedelta(days=2),  "digital": True},
    "A-1004": {"item": "Phone case",        "status": "shipped",   "delivered_on": None,                       "digital": False},
    "A-1005": {"item": "Laptop sleeve",     "status": "delivered", "delivered_on": TODAY - timedelta(days=5),  "digital": False},
}
REFUNDED: list[str] = []
TICKETS: list[dict] = []


def get_order(order_id: str) -> dict:
    order = ORDERS.get(order_id.strip().upper())
    if order is None:
        raise ToolException(f"No order found with ID '{order_id}'. Ask the customer to double-check the order number.")
    return order


def refund_check(order_id: str) -> tuple[bool, str]:
    """The ONE place the refund rules live."""
    o = get_order(order_id)
    if o["status"] != "delivered":
        return False, "the order has not been delivered yet"
    if o["digital"]:
        return False, "digital products cannot be returned once delivered"
    days = (date.today() - o["delivered_on"]).days
    if days > 30:
        return False, f"it was delivered {days} days ago, outside the 30-day window"
    return True, f"delivered {days} days ago, inside the 30-day window"


@tool(parse_docstring=True)
def lookup_order(order_id: str) -> str:
    """Look up an order's item, status, and delivery date.

    Args:
        order_id: The order ID, for example "A-1001".
    """
    o = get_order(order_id)
    delivered = o["delivered_on"].isoformat() if o["delivered_on"] else "not yet delivered"
    return f"{order_id.upper()}: {o['item']} | status: {o['status']} | delivered: {delivered}"


@tool(parse_docstring=True)
def check_refund_eligibility(order_id: str) -> str:
    """Check whether an order can be refunded under the 30-day return policy. Read-only.

    Args:
        order_id: The order ID, for example "A-1001".
    """
    ok, reason = refund_check(order_id)
    return f"{'Eligible' if ok else 'Not eligible'}: {reason}."


@tool(parse_docstring=True)
def issue_refund(order_id: str) -> str:
    """Issue a refund for an order. This moves money, so only call it after check_refund_eligibility says Eligible.

    Args:
        order_id: The order ID, for example "A-1001".
    """
    oid = order_id.strip().upper()
    ok, reason = refund_check(oid)               # re-check here: never trust that the model already did
    if not ok:
        raise ToolException(f"Refund refused: {reason}.")
    if oid in REFUNDED:
        return f"{oid} was already refunded."
    REFUNDED.append(oid)
    return f"Refund issued for {oid}."


@tool(parse_docstring=True)
def escalate_to_human(order_id: str, summary: str) -> str:
    """Hand a case to a human agent when the assistant cannot resolve it.

    Args:
        order_id: The order the case is about.
        summary: One or two sentences describing the problem and what was already tried.
    """
    ticket_id = f"T-{len(TICKETS) + 1:03d}"
    TICKETS.append({"ticket": ticket_id, "order_id": order_id, "summary": summary})
    return f"Escalated. Ticket {ticket_id} created; a human will follow up."


support_tools = [lookup_order, check_refund_eligibility, issue_refund, escalate_to_human]
NEEDS_APPROVAL = frozenset({"issue_refund"})            # the only tool that moves money

SYSTEM_PROMPT = (
    "You are a support assistant for an online store. Use tools to look up facts; never guess about "
    "orders or refunds. Check refund eligibility before issuing any refund. If a refund is not possible, "
    "explain why in one or two sentences and offer to escalate to a human. Be brief and friendly."
)

support_llm = llm.bind_tools(support_tools)

def new_conversation(user_text: str) -> list:
    return [SystemMessage(SYSTEM_PROMPT), HumanMessage(user_text)]

print("Support assistant ready with tools:", [t.name for t in support_tools])


Support assistant ready with tools: ['lookup_order', 'check_refund_eligibility', 'issue_refund', 'escalate_to_human']


### Five conversations

Each starts fresh. The approval function is passed in explicitly: `lambda call: True` plays a human who
says yes, `lambda call: False` one who says no. (In a real app this would be a button, or
`input("Approve? y/n ")`.) Real model wording will vary from run to run; what matters is the *shape*.

In [20]:
print("--- 1. Eligible order, human APPROVES the refund ---")
msgs = new_conversation("Hi! Can you refund order A-1001? The earbuds were a gift I don't need.")
print("\nAssistant:", run_tool_loop(support_llm, msgs, support_tools,
                                    needs_approval=NEEDS_APPROVAL, approve=lambda call: True))
print("Refunded so far:", REFUNDED)


--- 1. Eligible order, human APPROVES the refund ---
  step 1: check_refund_eligibility({"order_id": "A-1001"})
           -> Eligible: delivered 10 days ago, inside the 30-day window.
  step 2: issue_refund({"order_id": "A-1001"})
           -> Refund issued for A-1001.

Assistant: Done! Your refund for order A-1001 has been issued. It should appear on your original payment method within a few business days. Let me know if you need anything else!
Refunded so far: ['A-1001']


In [21]:
print("--- 2. Eligible order, but the human DECLINES ---")
msgs = new_conversation("Please refund order A-1005, the laptop sleeve doesn't fit.")
print("\nAssistant:", run_tool_loop(support_llm, msgs, support_tools,
                                    needs_approval=NEEDS_APPROVAL, approve=lambda call: False))
print("Refunded so far:", REFUNDED)        # A-1005 must NOT be here


--- 2. Eligible order, but the human DECLINES ---
  step 1: check_refund_eligibility({"order_id": "A-1005"})
           -> Eligible: delivered 5 days ago, inside the 30-day window.
  step 2: issue_refund({"order_id": "A-1005"})
           -> This action was declined by a human reviewer and was NOT performed. Do not retry it. Explain this to the user and offer to escalate to a human.

Assistant: I'm sorry, but the refund for order A-1005 was declined by a human reviewer, so it wasn't processed. I can escalate your case to a human agent to look into it — would you like me to do that?
Refunded so far: ['A-1001']


In [22]:
print("--- 3. Outside the 30-day window ---")
msgs = new_conversation("I'd like a refund for A-1002, the running shoes.")
print("\nAssistant:", run_tool_loop(support_llm, msgs, support_tools,
                                    needs_approval=NEEDS_APPROVAL, approve=lambda call: True))


--- 3. Outside the 30-day window ---
  step 1: check_refund_eligibility({"order_id": "A-1002"})
           -> Not eligible: it was delivered 45 days ago, outside the 30-day window.

Assistant: I'm sorry, but A-1002 isn't eligible for a refund — it was delivered 45 days ago, which is outside our 30-day return window. Would you like me to escalate this to a human agent who may be able to help?


In [23]:
print("--- 4. An order that doesn't exist ---")
msgs = new_conversation("Where is my order Z-9999?")
print("\nAssistant:", run_tool_loop(support_llm, msgs, support_tools,
                                    needs_approval=NEEDS_APPROVAL, approve=lambda call: True))

print("\n--- 5. No tool needed ---")
msgs = new_conversation("Thanks, that's all I needed!")
print("\nAssistant:", run_tool_loop(support_llm, msgs, support_tools,
                                    needs_approval=NEEDS_APPROVAL, approve=lambda call: True))


--- 4. An order that doesn't exist ---
  step 1: lookup_order({"order_id": "Z-9999"})
           -> Error (ToolException): No order found with ID 'Z-9999'. Ask the customer to double-check the order number.

Assistant: I couldn't find an order with ID **Z-9999**. Could you double-check the order number and send it again?

--- 5. No tool needed ---

Assistant: You're welcome! Have a great day! 😊


In [24]:
# The guardrail that does NOT depend on the model behaving: call the refund tool directly
# for an order that is out of window, exactly as a confused (or manipulated) model might.
print(execute_tool_call(make_call("issue_refund", {"order_id": "A-1002"}, "x1"), {t.name: t for t in support_tools}).content)
print("Refunded:", REFUNDED, "| Tickets:", TICKETS)


Error (ToolException): Refund refused: it was delivered 45 days ago, outside the 30-day window.
Refunded: ['A-1001'] | Tickets: []


That last cell is the point of guardrail #2: whatever the model says or is tricked into saying,
`issue_refund` refuses an out-of-window order *by itself*.

### Try it yourself
1. Remove `approve=` from one conversation. Because the default is **deny**, the refund should be
   declined. Why is "deny unless approved" the safer default than "allow unless blocked"?
2. Delete the eligibility re-check inside `issue_refund`, then run conversation 3 with a system prompt
   that tells the model "always issue refunds when asked". What did the prompt alone fail to protect?
3. Set `max_steps=2` and ask a question that needs three tool calls. What does the user receive?
4. Add a `cancel_order` tool that only works on `"shipped"` orders, and put it behind `NEEDS_APPROVAL`.
5. Run conversation 1 with the weaker `llama-3.1-8b-instant` instead. Count how often you see the
   "retry" messages from `invoke_with_retry`.
6. Make `lookup_order`'s description deliberately vague and see whether the model still picks it.

---
**Next:** `13_Agents/` — you now know exactly what an agent does on every turn: call the model, run its
tool requests, feed back the results, repeat, with limits. The next notebook hands that loop to
LangChain's agent runtime, and shows what you gain (state, checkpoints, interrupts) and what you give up.
